# Teste das funções de `data.py`

Este notebook serve para validar manualmente as funções de carga, preparação e separação dos dados antes da criação dos testes unitários.

In [1]:
from pathlib import Path
import sys

def find_project_root(start_path: Path) -> Path:
    """Encontra a raiz do projeto procurando pela pasta src/churn."""
    for path in [start_path, *start_path.parents]:
        if (path / "src" / "churn").exists():
            return path
    raise RuntimeError("Não foi possível encontrar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/Users/Lucas/Desktop/pos_tech/tech-challenge-fase1')

In [2]:
from data import (
    PREPROCESSED_DATA_PATH,
    NOTEBOOK_SELECTED_FEATURES,
    TARGET_COLUMN,
    add_engineered_features,
    clean_total_charges,
    load_csv_data,
    load_modeling_data,
    prepare_modeling_data,
    select_notebook_features,
    split_features_target,
    validate_required_columns,
)

PREPROCESSED_DATA_PATH

WindowsPath('C:/Users/Lucas/Desktop/pos_tech/tech-challenge-fase1/data/pre-processed/Telco_customer_churn_preprocessed.csv')

## 1. Carga do dataset pré-processado

In [3]:
df_raw = load_csv_data(PREPROCESSED_DATA_PATH)

print(f"Shape original: {df_raw.shape}")
df_raw.head()

Shape original: (7043, 58)


,CustomerID,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,...,Churn Reason_Lack of self-service on Website,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction
0,3668-QPYBK,1,90003,33.964131,-118.272783,2,53.85,108.15,1,86,...,0,0,0,0,0,0,0,0,0,0
1,9237-HQITU,1,90005,34.059281,-118.307420,2,70.70,151.65,1,67,...,0,0,0,1,0,0,0,0,0,0
2,9305-CDSKC,1,90006,34.048013,-118.293953,8,99.65,820.50,1,86,...,0,0,0,1,0,0,0,0,0,0
3,7892-POOKP,1,90010,34.062125,-118.315709,28,104.80,3046.05,1,84,...,0,0,0,1,0,0,0,0,0,0
4,0280-XJGEX,1,90015,34.039224,-118.266293,49,103.70,5036.30,1,89,...,0,0,0,0,0,0,0,0,0,0


In [5]:
validate_required_columns(df_raw)
print("Colunas obrigatórias encontradas com sucesso.")

Colunas obrigatórias encontradas com sucesso.


## 2. Limpeza e criação das features derivadas

In [6]:
df_clean = clean_total_charges(df_raw)

print(df_clean["Total Charges"].dtype)
df_clean[["Total Charges"]].head()

float64


,Total Charges
0,108.15
1,151.65
2,820.50
3,3046.05
4,5036.30


In [7]:
df_engineered = add_engineered_features(df_clean)

df_engineered[[
    "Monthly Charges",
    "Total Charges",
    "Tenure Months",
    "Engineered Monthly Charges",
    "charge_rel",
]].head()

,Monthly Charges,Total Charges,Tenure Months,Engineered Monthly Charges,charge_rel
0,53.85,108.15,2,2899.8225,-0.028259
1,70.70,151.65,2,4998.4900,0.115063
2,99.65,820.50,8,9930.1225,0.152012
3,104.80,3046.05,28,10983.0400,0.790643
4,103.70,5036.30,49,10753.6900,-0.897803


## 3. Seleção das 30 features do notebook 02

In [8]:
df_selected = select_notebook_features(df_engineered)

print(f"Shape selecionado: {df_selected.shape}")
print(f"Quantidade de features: {len(NOTEBOOK_SELECTED_FEATURES)}")
df_selected.head()

Shape selecionado: (7043, 31)
Quantidade de features: 30


,target,Tenure Months,Monthly Charges,Total Charges,Churn Score,Dependents_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_No internet service,Online Backup_No internet service,...,Churn Reason_Competitor made better offer,Churn Reason_Competitor offered higher download speeds,Churn Reason_Competitor offered more data,Churn Reason_Don't know,Churn Reason_Lack of self-service on Website,Churn Reason_Network reliability,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,charge_rel
0,1,2,53.85,108.15,86,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,-0.028259
1,1,2,70.70,151.65,67,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.115063
2,1,8,99.65,820.50,86,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.152012
3,1,28,104.80,3046.05,84,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.790643
4,1,49,103.70,5036.30,89,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,-0.897803


In [9]:
expected_columns = [TARGET_COLUMN, *NOTEBOOK_SELECTED_FEATURES]

assert df_selected.columns.tolist() == expected_columns
assert df_selected.shape[1] == 31

print("A seleção manteve a coluna alvo e as 30 features esperadas.")

A seleção manteve a coluna alvo e as 30 features esperadas.


## 4. Pipeline completo de preparação

In [10]:
df_modeling = prepare_modeling_data(df_raw)

print(f"Shape após preparação: {df_modeling.shape}")
df_modeling.head()

Shape após preparação: (7043, 31)


,target,Tenure Months,Monthly Charges,Total Charges,Churn Score,Dependents_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_No internet service,Online Backup_No internet service,...,Churn Reason_Competitor made better offer,Churn Reason_Competitor offered higher download speeds,Churn Reason_Competitor offered more data,Churn Reason_Don't know,Churn Reason_Lack of self-service on Website,Churn Reason_Network reliability,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,charge_rel
0,1,2,53.85,108.15,86,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,-0.028259
1,1,2,70.70,151.65,67,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.115063
2,1,8,99.65,820.50,86,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.152012
3,1,28,104.80,3046.05,84,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0.790643
4,1,49,103.70,5036.30,89,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,-0.897803


In [11]:
df_modeling_from_path = load_modeling_data()

assert df_modeling_from_path.equals(df_modeling)
print("load_modeling_data() gerou o mesmo resultado do pipeline manual.")

load_modeling_data() gerou o mesmo resultado do pipeline manual.


## 5. Separação entre X e y

In [12]:
X, y = split_features_target(df_modeling)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Distribuição do target:")
print(y.value_counts(normalize=True))

X.head()

X shape: (7043, 30)
y shape: (7043,)
Distribuição do target:
target
0    0.73463
1    0.26537
Name: proportion, dtype: float64


,Tenure Months,Monthly Charges,Total Charges,Churn Score,Dependents_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_No internet service,Online Backup_No internet service,Device Protection_No internet service,...,Churn Reason_Competitor made better offer,Churn Reason_Competitor offered higher download speeds,Churn Reason_Competitor offered more data,Churn Reason_Don't know,Churn Reason_Lack of self-service on Website,Churn Reason_Network reliability,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,charge_rel
0,2,53.85,108.15,86,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,-0.028259
1,2,70.70,151.65,67,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.115063
2,8,99.65,820.50,86,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.152012
3,28,104.80,3046.05,84,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.790643
4,49,103.70,5036.30,89,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-0.897803


In [ ]:
assert TARGET_COLUMN not in X.columns
assert len(X.columns) == 30
assert len(X) == len(y)

print("Separação X/y validada com sucesso.")

Separação X/y validada com sucesso.


: 